In [1]:
# Cell 1: Setup
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import mlflow
import mlflow.sklearn

# reproducibility
RND = 42
np.random.seed(RND)
random.seed(RND)

# MLflow local tracking directory (optional)
# mlflow.set_tracking_uri("file:///path/to/mlruns")  # uncomment to change
print("MLflow tracking URI:", mlflow.get_tracking_uri())


MLflow tracking URI: file:///C:/Users/z048724/notebook/MLops/week-8/mlruns


In [2]:
# !pip install mlflow


In [3]:
# Cell 2: Data poisoning utilities

def poison_data(X: np.ndarray, y: np.ndarray, fraction: float, mode: str = "feature", randomize_scale: float = 1.0):
    """
    Poison a fraction of the *training* dataset.
    - X: (N, D) features
    - y: (N,) labels
    - fraction: fraction of rows to poison (0.05 => 5%)
    - mode: "feature" (replace features with random numbers),
            "label" (flip labels randomly),
            "both" (do both)
    - randomize_scale: controls randomness scale relative to X std; higher -> more destructive

    Returns Xp, yp (copies)
    """
    assert 0 <= fraction <= 1.0
    n = X.shape[0]
    k = max(1, int(round(n * fraction)))
    idx = np.random.choice(n, k, replace=False)
    Xp = X.copy().astype(float)
    yp = y.copy().astype(int)

    if mode in ("feature", "both"):
        # generate random noise drawn from N(mean,std*randomize_scale)
        mu = np.mean(Xp, axis=0)
        sigma = np.std(Xp, axis=0) + 1e-6
        # produce random values per selected row
        noise = np.random.normal(loc=0.0, scale=1.0, size=(k, Xp.shape[1]))
        # scale into the feature domain
        random_vals = mu + (noise * sigma * randomize_scale)
        Xp[idx, :] = random_vals

    if mode in ("label", "both"):
        # flip labels to a random label different from true
        unique_labels = np.unique(yp)
        for i in idx:
            choices = [lab for lab in unique_labels if lab != int(yp[i])]
            yp[i] = np.random.choice(choices)

    return Xp, yp, idx


In [4]:
# Cell 3: Training & MLflow logging helper
RND = 42

def train_and_log_run(X_train, y_train, X_val, y_val,
                      poison_frac=0.0, poison_mode="feature", randomize_scale=1.0,
                      run_name=None, model_seed=RND, n_estimators=100):
    """
    Train a RandomForest on possibly-poisoned training data, evaluate on clean validation,
    and log to MLflow. Returns dict of metrics.
    """
    # Apply poisoning to training set only (we keep validation/test clean)
    Xp, yp, poisoned_idx = poison_data(X_train, y_train, fraction=poison_frac, mode=poison_mode,
                                       randomize_scale=randomize_scale)

    # Build pipeline
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(n_estimators=n_estimators, random_state=model_seed))
    ])

    # Start MLflow run
    with mlflow.start_run(run_name=run_name):
        # Log experiment params
        mlflow.log_param("poison_frac", poison_frac)
        mlflow.log_param("poison_mode", poison_mode)
        mlflow.log_param("randomize_scale", randomize_scale)
        mlflow.log_param("n_estimators", n_estimators)

        pipe.fit(Xp, yp)

        # Predict on clean validation
        y_pred = pipe.predict(X_val)
        acc = accuracy_score(y_val, y_pred)
        f1_macro = f1_score(y_val, y_pred, average="macro")
        class_report = classification_report(y_val, y_pred, output_dict=True)
        cm = confusion_matrix(y_val, y_pred)

        # Log metrics
        mlflow.log_metric("val_accuracy", float(acc))
        mlflow.log_metric("val_f1_macro", float(f1_macro))

        # Save confusion matrix figure as artifact
        fig, ax = plt.subplots(figsize=(5,4))
        cax = ax.matshow(cm, cmap=plt.cm.Blues)
        for (i, j), val in np.ndenumerate(cm):
            ax.text(j, i, int(val), ha="center", va="center", color="black")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"CM poison={poison_frac}, mode={poison_mode}")
        plt.colorbar(cax, ax=ax)
        plt.tight_layout()
        cm_path = f"confusion_poison_{poison_frac}_{poison_mode}.png"
        plt.savefig(cm_path)
        plt.close(fig)

        mlflow.log_artifact(cm_path)
        # Log model (sklearn)
        mlflow.sklearn.log_model(pipe, artifact_path="model")

        # Optionally log which rows were poisoned (as JSON)
        mlflow.log_text(json.dumps({"poisoned_indices": poisoned_idx.tolist()}), "poisoned_indices.json")

        # Return aggregated metrics
        run_info = {
            "poison_frac": poison_frac,
            "poison_mode": poison_mode,
            "val_accuracy": float(acc),
            "val_f1_macro": float(f1_macro),
            "confusion_matrix": cm.tolist()
        }
    return run_info


In [5]:
# Cell 4: Load Iris, train/test split, run experiments

data = load_iris()
X = data['data']
y = data['target']
feature_names = data['feature_names']
class_names = data['target_names']

# single held-out test set; we will train on train set (some of which will be poisoned)
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.20, random_state=RND, stratify=y)
# further split train/val for quicker validation (or use cross-val)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.15, random_state=RND, stratify=y_trainval)

print("Sizes: train", X_train.shape, "val", X_val.shape, "test", X_test.shape)

# experiment grid
poison_levels = [0.0, 0.05, 0.10, 0.50]   # 0.0 = clean baseline
mode = "feature"  # you can try "label" or "both"
randomize_scale = 2.0  # larger -> more destructive feature replacement

results = []
for frac in poison_levels:
    run_name = f"poison_{int(frac*100)}pct_{mode}"
    info = train_and_log_run(X_train, y_train, X_val, y_val,
                             poison_frac=frac,
                             poison_mode=mode,
                             randomize_scale=randomize_scale,
                             run_name=run_name,
                             n_estimators=200)
    results.append(info)
    print(f"Completed: {run_name} → acc={info['val_accuracy']:.4f}, f1_macro={info['val_f1_macro']:.4f}")

# Evaluate best run on the held-out test set for demonstration (e.g., baseline vs 50%)
print("\nNow evaluate baseline (0% poison) and 50% poison model on the clean holdout test set.")
# Helper to load model from MLflow run or re-train (we will re-train here deterministically for simplicity)
baseline = train_and_log_run(X_train, y_train, X_test, y_test, poison_frac=0.0, poison_mode=mode, run_name="baseline_final_eval")
poison50 = train_and_log_run(X_train, y_train, X_test, y_test, poison_frac=0.5, poison_mode=mode, run_name="poison50_final_eval")
print("Baseline test acc:", baseline['val_accuracy'], "poison50 test acc:", poison50['val_accuracy'])


Sizes: train (102, 4) val (18, 4) test (30, 4)


C:\Users\z048724\notebook\venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2025/11/16 17:52:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 17:52:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Completed: poison_0pct_feature → acc=0.9444, f1_macro=0.9441


2025/11/16 17:52:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 17:52:41 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Completed: poison_5pct_feature → acc=0.9444, f1_macro=0.9441


...

Completed: poison_10pct_feature → acc=0.9444, f1_macro=0.9441


2025/11/16 17:52:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 17:52:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Completed: poison_50pct_feature → acc=0.9444, f1_macro=0.9441

Now evaluate baseline (0% poison) and 50% poison model on the clean holdout test set.


2025/11/16 17:52:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 17:53:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/16 17:53:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 17:53:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Baseline test acc: 0.9666666666666667 poison50 test acc: 0.9666666666666667


In [6]:
# Cell 5: summarize results table
df_res = pd.DataFrame(results)
display(df_res)
# Save as artifact

df_res.to_csv("poisoning_results_summary.csv", index=False)
print("Saved summary to poisoning_results_summary.csv")


,poison_frac,poison_mode,val_accuracy,val_f1_macro,confusion_matrix
0,0.00,feature,0.944444,0.944056,"[[6, 0, 0], [0, 5, 1], [0, 0, 6]]"
1,0.05,feature,0.944444,0.944056,"[[6, 0, 0], [0, 5, 1], [0, 0, 6]]"
2,0.10,feature,0.944444,0.944056,"[[6, 0, 0], [0, 5, 1], [0, 0, 6]]"
3,0.50,feature,0.944444,0.944056,"[[6, 0, 0], [0, 5, 1], [0, 0, 6]]"


Saved summary to poisoning_results_summary.csv
